In [ ]:
import sys
sys.path.append('../')
import time

import h5py
import epics
import numpy as np
import matplotlib.pyplot as plt

from siriuspy.devices import CAXCtrl
from caxscripts.h5file import HDF5File

Initialize cax control of siriuspy devices

In [42]:
CAX = CAXCtrl()

# Scan

Parameters

In [ ]:
STEP = 0.0001 # [mm]
SCALE = 1     # [um/px]
DELAY = 5     # [s]
MAXERRORCOUNT = 5

Functions

In [ ]:
def dvfs_acquire():
    CAX.dvf_A1.cmd_acquire_on()
    CAX.dvf_B1.cmd_acquire_on()

def current_parameters():
    return {
        'slit1': {
            'top': CAX.slit_A1.top_pos,
            'bottom': CAX.slit_A1.bottom_pos,
            'left': CAX.slit_A1.left_pos,
            'right': CAX.slit_A1.right_pos
        },
        'dvf1': {
            'expo_time': CAX.dvf_A1.acquisition_time
        },
        'dvf2': {
            'expo_time': CAX.dvf_A1.acquisition_time,
            'z_pos': CAX.dvf_B1.z_pos
        },
        'mirror': {
            'ry': CAX.mirror.ry_pos,
            'tx': CAX.mirror.tx_pos,
            'y1': CAX.mirror.y1_pos,
            'y2': CAX.mirror.y2_pos,
            'y3': CAX.mirror.y3_pos,
            'photocollector': CAX.mirror.photocurrent_signal
        }
    }

def fwhm(data):
    threshold = 0.5*np.max(data)
    mask = data > threshold
    return np.sum(mask) * SCALE

def peak(data):
    return np.max(data)

def position(data):
    return np.argmax(data) * SCALE

def move_all_slits(top, bottom, left, right):
    CAX.slit_A1.move_top(value=top)
    CAX.slit_A1.move_bottom(value=bottom)
    CAX.slit_A1.move_left(value=left)
    CAX.slit_A1.move_right(value=right)

def open_all_slits():
    move_all_slits(top=19.7-0.04,
                   bottom=35.8,
                   left=43.6-0.04,
                   right=47.2)

def move(ry_pos,delay=DELAY):
    CAX.mirror.ry_pos = ry_pos
    time.sleep(delay)

def move_step(step=STEP,delay=DELAY):
    pos = CAX.mirror.ry_pos + step
    move(ry_pos=pos,delay=delay)


def get_image(dvf):

    count = 0
    while count < MAXERRORCOUNT:
        try:
            if not dvf.acquisition_status:
                dvf.cmd_acquire_on()
            return dvf.image
        except Exception as err:
            print(f" WARNING. When trying to fetch image from DVF1: {err} ")
            time.sleep(2)
            count += 1
            if count < MAXERRORCOUNT:
                print("\n Repeating the procedure...\n")
            else:
                raise Exception("Client exception")



Initial state

In [ ]:
local_time = time.localtime()
formatted_time = time.strftime("%Y%m%d-%H%M%S", local_time)
formatted_date = time.strftime("%Y%m%d", local_time)

formatted_time, formatted_date

In [ ]:
ANALYSIS = 'ry'

In [ ]:
parameters0 = current_parameters()

states_dir = '../states/'
state_name = f'{formatted_time}-{ANALYSIS}.txt'

with open(states_dir+state_name, 'w') as f:
    f.write(str(parameters0))

print(parameters0)

Initialize scan file

In [ ]:
scaname = f'scan_ry_{formatted_date}.h5'
datadir  = '/home/ids/data/'
direc    = f'{formatted_date}-Mirror-Ry/'

In [ ]:
file = HDF5File(filename=scaname,filedir=datadir+direc)

file.save_metadata(metadata_dict={
    'ry': CAX.mirror.ry_pos
})

Execution

In [210]:
amplitude_ry = 0.343029 - 0.3336945
amplitude_ry * 1e3

9.334499999999968

In [ ]:
open_all_slits()

In [227]:
ry0 = CAX.mirror.ry_pos

In [ ]:
positions1 = ry0 + np.linspace(0,amplitude_ry,101)[1:]
positions2 = positions1.copy()[-2::-1]
positions = np.hstack([positions1,positions2])

In [ ]:
t0 = time.time()


for i, pos in enumerate(positions):
    print(f'{i}/{len(positions)-1}')

    #!
    #todo: deslocar feixe ate a borda. como está agora é
    #todo: a partir do valor inicial, mas deve ser a partir da borda

    move(ry_pos=pos)

    img1 = get_image(dvf=CAX.dvf_A1)
    expotime1 = CAX.dvf_A1.exposure_time
    img2 = get_image(dvf=CAX.dvf_B1)
    expotime2 = CAX.dvf_B1.exposure_time

    scan = f'scan-{i:04d}'
    scanmetadata = {
        'ry_pos': CAX.mirror.ry_pos,
        'photocollector': CAX.mirror.photocurrent_signal
    }
    file.save_group(grpname=scan, grpmetadata=scanmetadata)
    file.save_dataset(grpname=scan, dsetname='dvf1', 
                      dsetmetadata={'expo_time':expotime1}, dsetdata=img1)
    file.save_dataset(grpname=scan, dsetname='dvf2', 
                      dsetmetadata={'expo_time':expotime2}, dsetdata=img2)


t1 = time.time()

print()
print(f'elapsed time [min]: {(t1-t0)/60}')

0/198
1/198
2/198
3/198
4/198
5/198
6/198
7/198
8/198
9/198
10/198
11/198
12/198
13/198
14/198
15/198
16/198
17/198
18/198
19/198
20/198
21/198
22/198
23/198
24/198
25/198
26/198
27/198
28/198
29/198
30/198
31/198
32/198
33/198
34/198
35/198
36/198
37/198
38/198
39/198
40/198
41/198
42/198
43/198
44/198
45/198
46/198
47/198
48/198
49/198
50/198
51/198
52/198
53/198
54/198
55/198
56/198
57/198
58/198
59/198
60/198
61/198
62/198
63/198
64/198
65/198
66/198
67/198
68/198
69/198
70/198
71/198
72/198
73/198
74/198
75/198
76/198
77/198
78/198
79/198
80/198
81/198
82/198
83/198
84/198
85/198
86/198
87/198
88/198
89/198
90/198
91/198
92/198
93/198
94/198
95/198
96/198
97/198
98/198
99/198
100/198
101/198
102/198
103/198
104/198
105/198
106/198
107/198
108/198
109/198
110/198
111/198
112/198
113/198
114/198
115/198
116/198
117/198
118/198
119/198
120/198
121/198
122/198
123/198
124/198
125/198
126/198
127/198
128/198
129/198
130/198
131/198
132/198
133/198
134/198
135/198
136/198
137/198
138/19